# TP3.1 — Implementing DDPM and Sampling Process

**Practical Exercise — Introduction to Deep Generative Models**  
For each practical exercise (TP), please work in groups of two or three. Then, create a **private GitHub repository** and add me (GitHub: **arthur-75**) to your project.

**Instructions:** Complete the `xxx` placeholders. Questions and examples are kept as in the source material. Some code intentionally cannot run before completion. The `diffusion_utilities` module and trained weights must be supplied separately.

## 1. Understanding the Diffusion Model

Denoising Diffusion Probabilistic Models (DDPM) generate images by reversing a noise diffusion process. The model learns to denoise images step by step, moving from pure noise to a meaningful image.

### Mathematical Foundation

The forward process gradually adds Gaussian noise:

$$q(x_t\mid x_{t-1})=\mathcal{N}(x_t;\sqrt{1-\beta_t}\,x_{t-1},\beta_t I)$$

where:
- $x_t$ is the noised image at step $t$;
- $\beta_t$ is the variance schedule that controls noise addition;
- $I$ is the identity matrix.

The reverse process aims to remove noise:

$$p_\theta(x_{t-1}\mid x_t)=\mathcal{N}(x_{t-1};\mu_\theta(x_t,t),\Sigma_\theta(x_t,t))$$

where $\mu_\theta(x_t,t)$ is predicted by a neural network.

### Import necessary packages

In [ ]:
# get the course files from the Drive "cours 4 DM" (gdown is already installed on Colab)
# or upload them by hand: diffusion_utilities.py, the models/ folder and the data/ folder
import os
import gdown

if not os.path.exists("diffusion_utilities.py"):
    gdown.download(id="1xN0YQfdAdYjF2mpAftCpBBYfxMdwgjpN", output="diffusion_utilities.py")
if not os.path.exists("models"):
    gdown.download_folder(id="1u9Xl23P1C8TNKa6SZxnx4XS6BaurwVoX", output="models")
print(os.listdir("models"))

Downloading...
From (original): https://drive.google.com/uc?id=1xN0YQfdAdYjF2mpAftCpBBYfxMdwgjpN
From (redirected): https://drive.google.com/uc?id=1xN0YQfdAdYjF2mpAftCpBBYfxMdwgjpN&confirm=t&uuid=531d8ac4-525a-432d-858c-e417f3234d70
To: /content/diffusion_utilities.py
100%|██████████| 9.93k/9.93k [00:00<00:00, 8.73MB/s]
Retrieving folder contents


Processing file 1RncKsQi1mmrktyo6c9jLWELW586VJf9U context_model_0.pth
Processing file 1OtOHPYEIeu3-CEhnY15ZZDifMd44C8zv context_model_4.pth
Processing file 1ySVxtx4sK4EMAIZ0cddXHzR9hA6RK4oB context_model_8.pth
Processing file 1bRl2n232PxmZkWdiniTCtWT668fGd005 context_model_12.pth
Processing file 1wbdzTWVTQeUo2mK9H3Ip_nSFRpfnSMhO context_model_16.pth
Processing file 1obwI64lar7q_TiO9LqPVIEEc4NkSBuuz context_model_20.pth
Processing file 1s47l76eoYovZsl3P_qLK2D5qTXEaDJFa context_model_24.pth
Processing file 1HtvZ6VkfaCkSizjVqs69j53WVaKhe6s6 context_model_28.pth
Processing file 1qYpKkZk6mt1r-uXNglfOixPZLoey5if_ context_model_31.pth
Processing file 1QfhraKeTtOTAIBjCExA4zWUPMf55H9z8 context_model_trained.pth
Processing file 1miK3sxBOdOUlvi25NEW-tYnnJruhB1P4 model_0.pth
Processing file 1rF907kKhk25QVt_hVuhTcFG26GeA8Fg2 model_4.pth
Processing file 1XHXFOr29JvEL4cBYoIUWqr2gtVA5cqYP model_8.pth
Processing file 1pnf1bm7t4ouaPIP6zqAxFhdb-38aRKwJ model_12.pth
Processing file 1Tsv8Gq2Zhr9ruC3yhnNnkS

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1RncKsQi1mmrktyo6c9jLWELW586VJf9U
To: /content/models/context_model_0.pth
100%|██████████| 5.99M/5.99M [00:00<00:00, 54.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OtOHPYEIeu3-CEhnY15ZZDifMd44C8zv
To: /content/models/context_model_4.pth
100%|██████████| 5.99M/5.99M [00:00<00:00, 36.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ySVxtx4sK4EMAIZ0cddXHzR9hA6RK4oB
To: /content/models/context_model_8.pth
100%|██████████| 5.99M/5.99M [00:00<00:00, 69.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bRl2n232PxmZkWdiniTCtWT668fGd005
To: /content/models/context_model_12.pth
100%|██████████| 5.99M/5.99M [00:00<00:00, 81.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wbdzTWVTQeUo2mK9H3Ip_nSFRpfnSMhO
To: /content/models/context_model_16.pth
100%|██████████| 5.99M/5.99M [00:00<00:00, 67.2MB/s

['context_model_28.pth', 'model_12.pth', 'model_28.pth', 'context_model_20.pth', 'context_model_24.pth', 'model_0.pth', 'model_24.pth', 'model_31.pth', 'model_20.pth', 'model_8.pth', 'model_4.pth', 'context_model_0.pth', 'model_trained.pth', 'context_model_12.pth', 'context_model_4.pth', 'model_16.pth', 'context_model_8.pth', 'context_model_16.pth', 'context_model_trained.pth', 'context_model_31.pth']



Download completed


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from diffusion_utilities import ResidualConvBlock, UnetUp, UnetDown, EmbedFC, plot_sample

# Hyperparameters
# diffusion hyperparameters
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

# network hyperparameters
device = torch.device("cuda:0" if torch.cuda.is_available() else torch.device('cpu'))
n_feat = 64  # 64 hidden dimension feature
n_cfeat = 5  # context vector is of size 5
height = 16  # 16x16 image

## 2. Constructing the Noise Schedule

The noise schedule determines how noise is added over $T$ steps.

### Defining the Noise Parameters

Linear schedule: A common choice is a linear increase from a small value to a larger value:

$$\beta_t=\beta_1+(\beta_2-\beta_1)\frac{t}{T}$$

In [ ]:
# construct DDPM noise schedule
b_t = beta1 + ((beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device))

**Cumulative product of alphas:** Defines how much original image information remains at time $t$; $\bar\alpha_t$ helps efficiently sample from any step:

$$\alpha_t=1-\beta_t,\qquad \bar\alpha_t=\prod_{s=1}^{t}\alpha_s.$$

In [ ]:
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1

## 3. The U-Net Architecture for Noise Prediction

*Nothing to do here; just try to understand.*

A U-Net is used to predict noise at each timestep.

### Architecture Breakdown

1. **Down-sampling Path (Encoder):** Extracts features from the input image using convolutional layers and successively reduces spatial resolution.
2. **Bottleneck:** Maps high-dimensional features to a lower-dimensional representation; embeds both time and contextual information (time tracks noise levels, context enables conditional generation).
3. **Up-sampling Path (Decoder):** Increases resolution using transposed convolutions, combines feature maps from the encoder (skip connections), and outputs predicted noise.

**Original U-Net architecture illustrations (from the handout):** see the figure in the original notebook / handout.

For an input image $x_t$, the network predicts noise $\epsilon_\theta(x_t,t)$.

In [ ]:
class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat=256, n_cfeat=10, height=28):  # cfeat - context features
        super(ContextUnet, self).__init__()

        # number of input channels, number of intermediate feature maps and number of classes
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_cfeat = n_cfeat
        self.h = height  # assume h == w. must be divisible by 4, so 28,24,20,16...

        # Initialize the initial convolutional layer
        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)

        # Initialize the down-sampling path of the U-Net with two levels
        self.down1 = UnetDown(n_feat, n_feat)        # down1 #[10, 256, 8, 8]
        self.down2 = UnetDown(n_feat, 2 * n_feat)    # down2 #[10, 256, 4, 4]

        # original: self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())
        self.to_vec = nn.Sequential(nn.AvgPool2d((4)), nn.GELU())

        # Embed the timestep and context labels with a one-layer fully connected neural network
        self.timeembed1 = EmbedFC(1, 2*n_feat)
        self.timeembed2 = EmbedFC(1, 1*n_feat)
        self.contextembed1 = EmbedFC(n_cfeat, 2*n_feat)
        self.contextembed2 = EmbedFC(n_cfeat, 1*n_feat)

        # Initialize the up-sampling path of the U-Net with three levels
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, self.h//4, self.h//4),  # up-sample
            nn.GroupNorm(8, 2 * n_feat),  # normalize
            nn.ReLU(),
        )
        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)

        # Initialize the final convolutional layers to map to same number of channels as input
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),  # reduce number of feature maps
            nn.GroupNorm(8, n_feat),  # normalize
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),  # map to same number of channels as input
        )

    def forward(self, x, t, c=None):
        """
        x : (batch, n_feat, h, w) : input image
        t : (batch, n_cfeat) : time step
        c : (batch, n_classes) : context label
        """
        # x is the input image, c is the context label, t is the timestep,
        # context_mask says which samples to block the context on
        x = self.init_conv(x)
        down1 = self.down1(x)          #[10, 256, 8, 8]
        down2 = self.down2(down1)      #[10, 256, 4, 4]
        hiddenvec = self.to_vec(down2)

        # mask out context if context_mask == 1
        if c is None:
            c = torch.zeros(x.shape[0], self.n_cfeat).to(x)

        # embed context and timestep
        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1*up1 + temb1, down2)
        up3 = self.up2(cemb2*up2 + temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out

# construct model
nn_model = ContextUnet(in_channels=3, n_feat=n_feat, n_cfeat=n_cfeat, height=height).to(device)

In [ ]:
# helper function; removes the predicted noise (but adds some noise back in to avoid collapse)
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise

## 4. The Sampling Process

Once trained, the model can generate images by reversing the noise process.

### Load the model

In [ ]:
# load in model weights and set to eval mode
save_dir = './models/'
nn_model.load_state_dict(torch.load(f"{save_dir}/model_trained.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

Loaded in Model


### DDPM Sampling Algorithm

1. Initialize $x_T\sim\mathcal N(0,I)$ (pure noise).
2. For $t=T$ to $1$:
   - Predict noise $\epsilon_\theta(x_t,t)$.
   - Compute the mean.
   - Add noise.

Complete the remaining placeholders in the code below.

In [ ]:
# sample using standard algorithm
@torch.no_grad()
def sample_ddpm(n_sample, save_rate=20):
    # x_T ~ N(0, 1), sample initial noise
    samples = torch.randn(n_sample, 3, height, height).to(device)

    # array to keep track of generated steps for plotting
    intermediate = []
    for i in range(timesteps, 0, -1):
        print(f'sampling timestep {i:3d}', end='\r')

        # reshape time tensor
        t = torch.tensor([i / timesteps])[:, None, None, None].to(device)

        # sample some random noise to inject back in. For i = 1, don't add back in noise
        z = torch.randn_like(samples) if i > 1 else 0

        eps = nn_model(samples, t)    # predict noise e_(x_t,t)
        samples = denoise_add_noise(samples, i, eps, z)
        if i % save_rate == 0 or i == timesteps or i < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate

**Additional helper function from the handout (unfinished version).**

In [ ]:
# helper function; removes the predicted noise (but adds some noise back in to avoid collapse)
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise

## 5. Visualization of the Sampling Process

In [ ]:
# visualize samples
plt.clf()
samples, intermediate_ddpm = sample_ddpm(32)
animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, save_dir, "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())

<Figure size 640x480 with 0 Axes>

## 6. Incorrect Sampling Process without Adding the “Extra Noise”

Compare the correct sampling procedure against the following implementation, which does not inject extra noise.

In [ ]:
# incorrectly sample without adding in noise
@torch.no_grad()
def sample_ddpm_incorrect(n_sample):
    # x_T ~ N(0, 1), sample initial noise
    samples = torch.randn(n_sample, 3, height, height).to(device)

    # array to keep track of generated steps for plotting
    intermediate = []
    for i in range(timesteps, 0, -1):
        print(f'sampling timestep {i:3d}', end='\r')

        # reshape time tensor
        t = torch.tensor([i / timesteps])[:, None, None, None].to(device)
        # don't add back in noise
        z = 0
        eps = nn_model(samples, t)  # predict noise e_(x_t,t)
        samples = denoise_add_noise(samples, i, eps, z)
        if i % 20 == 0 or i == timesteps or i < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate

# visualize samples
plt.clf()
samples, intermediate = sample_ddpm_incorrect(32)
animation = plot_sample(intermediate, 32, 4, save_dir, "ani_run", None, save=False)
HTML(animation.to_jshtml())

<Figure size 640x480 with 0 Axes>

### Question (to answer)

**Explain why we need to add extra noise and what happens if you do not add it.**

*Your answer:*

The reverse step $p_\theta(x_{t-1}\mid x_t)$ is a Gaussian distribution, so to sample from it we take the mean (computed with the predicted noise) **and** add $\sqrt{\beta_t}\,z$ with $z\sim\mathcal N(0,I)$. Also, the network was trained on images that have exactly the noise level of step $t$. If we only remove noise, $x_{t-1}$ is less noisy than what the model expects at the next step, and the errors add up over the 500 steps.

Without the extra noise the sampling becomes deterministic and every image goes toward an "average" image: we get blurry blobs with almost no details, and the 32 samples look similar (we see it in the second animation).